### Improved RAG
- [x] PDF URL loader
- [x] new chunking method
- [x] Semantic chunking  Resursive text already semantic chunking
- [x] data preparation
- [x] establish baseline benchmark
- [ ] Hybrid search: combine BM25 (keywords) + vector (semantic) + MMR (diversity).
- [x] Rerank retrieval result by LLM
- [ ] Rewrite query

In [1]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
import numpy as np
from sklearn.manifold import TSNE
import glob
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import chromadb
from langchain_chroma import Chroma


load_dotenv(override=True)

True

In [5]:
import gradio as gr
print(gr.__version__)
# Gradio 6 onwards will generate hisotry in the format of [{"role": "user", "content": "..."}, ...]
# Gradio 5 and below will generate history in the format of [{"text": "..."}, ...]


5.49.1


In [6]:
DB_NAME = "chroma_db"
DATA_DIR = '../data' 
MODEL = "gpt-4o-mini"
TOP_K = 5

In [4]:
# load docs

def load_document_md() -> str:
    documents = []

    doc_type = os.path.basename(DATA_DIR) # get the name of the folder. employees, contracts, products, company
    loader = DirectoryLoader(DATA_DIR, glob="*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        doc.metadata["source"] = doc.metadata.get("source").split("/")[-1]
        documents.append(doc) # Has to be Document object !!!
    print(f"loaded {len(documents)} md documents")
    return documents


In [16]:
# load docs

def _parse_md_frontmatter(content: str):
    """
    Parse simple YAML-like frontmatter:

    ---
    key: value
    tags: [a, b, c]
    ---
    """
    lines = content.splitlines()
    if not lines or lines[0].strip() != "---":
        return {}, content

    meta_lines = []
    i = 1
    while i < len(lines) and lines[i].strip() != "---":
        meta_lines.append(lines[i])
        i += 1

    if i >= len(lines):
        # no closing ---
        return {}, content

    body = "\n".join(lines[i + 1:])

    metadata = {}
    for line in meta_lines:
        stripped = line.strip()
        if not stripped or ":" not in stripped:
            continue
        key, val = stripped.split(":", 1)
        key = key.strip()
        val = val.strip()
        
        metadata[key] = val.strip().strip("'\"")

    return metadata, body.lstrip("\n")


def load_document_md() -> str:
    documents = []

    # Load all markdown recursively under DATA_DIR
    loader = DirectoryLoader(
        DATA_DIR,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    folder_docs = loader.load()

    for doc in folder_docs:
        raw_source = doc.metadata.get("source", "")
        source_path = raw_source

        # Parse and strip frontmatter
        frontmatter_meta, body = _parse_md_frontmatter(doc.page_content)
        if frontmatter_meta:
            doc.page_content = body
            for k, v in frontmatter_meta.items():
                doc.metadata[k] = v

        # doc_type: prefer frontmatter "type", else parent folder name
        if "type" in doc.metadata:
            doc.metadata["doc_type"] = doc.metadata["type"]
        else:
            parent = os.path.basename(os.path.dirname(source_path))
            doc.metadata["doc_type"] = parent or os.path.basename(DATA_DIR)

        # source: just the filename
        if raw_source:
            doc.metadata["source"] = os.path.basename(raw_source)

        documents.append(doc)  # Has to be Document object !!!

    print(f"loaded {len(documents)} md documents")
    return documents

### Improvement 1: PDF/URL Loader

In [14]:
from langchain_community.document_loaders import PlaywrightURLLoader, PyPDFLoader, UnstructuredHTMLLoader
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

def load_document_pdf() -> str:
    documents = []
    doc_type = os.path.basename(DATA_DIR)
    loader = DirectoryLoader(DATA_DIR, glob="*.pdf", loader_cls=PyPDFLoader)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        doc.metadata["source"] = doc.metadata.get("source").split("/")[-1]
        documents.append(doc)
        print(f"loaded {len(documents)} pdf documents")
        
    return documents

from langchain_community.document_loaders import WebBaseLoader
from urllib.parse import urlparse
from pathlib import Path

LINKEDIN_PROMPT = """
You are a text cleaning and summarization assistant. 

I will provide you with a raw text dump from a LinkedIn page. Your task is to:

1. Remove all unnecessary website text, boilerplate, and repeated login/signup prompts.
2. Remove all \n, excessive spaces, and strange characters.
3. Extract only the **personal profile information** of the user, including:
   - Name
   - Headline / Summary
   - Location
   - Connections / Followers (optional)
   - Current Experience (Job Title + Company + Years)
   - Past Experience (Job Title + Company + Years)
   - Education (School + Degree + Years)
   - Certifications or Courses
   - Languages
   - Projects or notable achievements
4. Present the information in a **clean, readable, structured format**, like:
   
   Name: ...
   Headline: ...
   Location: ...
   Current Experience: ...
   Past Experience: ...
   Education: ...
   Certifications: ...
   Languages: ...
   Projects: ...
   
5. Ignore any text that is not part of the user profile (e.g., "Sign in", "Join now", LinkedIn navigation, footer, ads, etc.).
"""

"""TODO: better way to handle html content, can not use "PlaywrightURLLoader since not supported in py3.12"""
def load_document_url():
    """
    Load documents from a file containing URLs and save them as text files
    """
    documents = []

    urls_file = Path(DATA_DIR) / "urls.txt"
    if not urls_file.exists():
        return documents

    with open(urls_file) as f:
        urls = [u.strip() for u in f.readlines() if u.strip()]

    if not urls:
        return documents

    loader = WebBaseLoader(urls)
    loaded_docs = loader.load()

    for doc in loaded_docs:
        doc.page_content = doc.page_content.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
        parsed = urlparse(doc.metadata.get("source", ""))
        doc.metadata["doc_type"] = "url"
        doc.metadata["source"] = parsed.netloc
        if doc.metadata.get("source") == "www.linkedin.com":
            llm = ChatOpenAI(model=MODEL, temperature=0)
            messages = [SystemMessage(content=LINKEDIN_PROMPT.format(raw_linkedin_text=doc.page_content))]
            messages.append(HumanMessage(content=doc.page_content))
            doc.page_content = llm.invoke(messages).content
        doc.metadata["full_url"] = doc.metadata.get("source")
        documents.append(doc)
        with open(Path(DATA_DIR) / (doc.metadata.get("source") + ".txt"), "w") as f:
            f.write(doc.page_content)

    print(f"Loaded {len(documents)} URL documents")
    return documents

In [15]:
load_document_pdf()

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)


loaded 1 pdf documents


[Document(metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260122073854Z00'00'", 'title': 'Resume_LI_BEIJI', 'author': 'beiji M4', 'moddate': "D:20260122073854Z00'00'", 'source': 'Resume_LI_BEIJI.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'doc_type': 'data'}, page_content="LI BEIJI +65 8432 9134    Linkedin    libeiji08121999@gmail.com  Github   Portfolio SUMMARY Passionate Software Engineer and AI Innovator with a track record of bridging cutting-edge AI solutions with a solid software engineering base. Skilled in designing and leading full-stack platforms and intelligent automation systems, drive real-world impact and unlock actionable AI insights.   WORK EXPERIENCE Software Engineer Aug 2023 - Present United Overseas Bank  • Led adoption and production rollout of n8n, empowering 10+ teams to build customized automation workflow or agentic solution with 10k SGD saved per month. • Designed and implemented Pr

In [16]:
load_document_url()

Loaded 2 URL documents


[Document(metadata={'source': 'portfolio-beiji.vercel.app', 'title': "Beiji's Blog", 'description': 'Personal thoughts and insights on technology, design, and life.', 'language': 'en', 'doc_type': 'url', 'full_url': 'portfolio-beiji.vercel.app'}, page_content="Beiji's Blogʵʵ.mp3 - Deca JoinsLi BeijiHomeContactYour browser does not support the video tag...WelcomeLi Beiji/˥Hi! This incredible journey begins with obsessing over the PC games and getting addicted to the whole computer world in the end. Now I'm a passionate Software Engineer and AI innovator, focused on creating meaningful digital experiences.BACK TO HOMEFeatured ProjectsSep 15, 2023Terminal-Based TransactionManagementSystem for BankA terminal-based banking system built with Java and PostgreSQL, featuring role-based access, secure authentication, transaction management, and a user-friendly CLI interface with audio feedback.Read MoreBackendOct 15, 2023Role-Based Banking Management SystemA secure banking management system with

In [17]:
documents = load_document_md() 
#documents.extend(load_document_url())
#documents.extend(load_document_pdf())

print(f"loaded {len(documents)} documents")


loaded 12 md documents
loaded 12 documents


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")
print(f"Second chunk:\n\n{chunks[1]}")
print(f"Last chunk:\n\n{chunks[-1]}")

AttributeError: 'MarkdownHeaderTextSplitter' object has no attribute 'split_documents'

In [40]:
# markdown splitter
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Two-level split: ## and ### — matches NUS-Master-Project (e.g. Pipeline A/B, Challenges subsections)
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("##", "section"),
        ("###", "subsection"),
    ]
)

chunked_docs = []
for doc in documents:
    # Split the markdown body into section chunks
    section_chunks = markdown_splitter.split_text(doc.page_content)
    for c in section_chunks:
        # Carry over original metadata, plus any header info (`section`)
        new_meta = dict(doc.metadata)
        # c.metadata may contain {"section": "1. Overview"} etc.
        section_title = c.metadata.get("section")
        project_title = doc.metadata.get("title")
        if section_title:
            c.page_content = f"project title: {project_title}\n\nsection: {section_title}\n\n{c.page_content}"
        new_meta.update(c.metadata)
        chunked_docs.append(
            Document(page_content=c.page_content, metadata=new_meta)
        )

chunks = chunked_docs

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk metadata:\n\n{chunks[0].metadata}")
print(f"First chunk:\n\n{chunks[0]}")
print(f"Second chunk:\n\n{chunks[1]}")
print(f"Last chunk:\n\n{chunks[-1]}")

Divided into 124 chunks
First chunk metadata:

{'source': 'hobby.md', 'type': 'personal', 'title': 'Hobbies & Personal Development', 'importance': 'low', 'year': '2026', 'tags': '[hobby, basketball, diving, weightlifting, leadership, discipline, stress-management, growth-mindset]', 'doc_type': 'personal', 'section': '1. Basketball'}
First chunk:

page_content='project title: Hobbies & Personal Development

section: 1. Basketball

Context:
Beiji has been playing basketball since middle school and continues recreationally today.  
University Experience:
- Served as captain of the Transportation Department basketball team at Southeast University.
- Led:
- Training sessions
- Match preparation
- Tactical coordination  
Skills Developed:
- Leadership under pressure
- Team coordination
- Fast decision-making  
Current Role in Life:
- Stress relief after work
- Maintaining physical fitness  
Personal Insight:
Basketball helped build early confidence and collaborative mindset that later transl

In [41]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings,collection_name="my_collection").delete_collection()

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
    collection_name="my_collection"
)

print(f"Vectorstore created with {vector_store._collection.count()} documents")

collection = vector_store._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

Vectorstore created with 124 documents
There are 124 vectors with 3,072 dimensions in the vector store


In [42]:
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings, collection_name="my_collection")
retriever = vectorstore.as_retriever()
retriever.invoke("What is Beiji's current job based on CV?")

[Document(id='21654dce-382f-44bb-81fb-3995a45840ed', metadata={'year': '2026', 'section': 'LI BEIJI', 'source': 'cv.md', 'title': 'Resume', 'type': 'CV', 'doc_type': 'CV', 'tags': '[cv, resume, ai-platform, devops, full-stack, llm, ai, uob, nus, career-summary, skills]', 'importance': 'high'}, page_content='project title: Resume\n\nsection: LI BEIJI\n\nPhone: +65 8432 9134\nEmail: libeiji08121999@gmail.com\nLinkedIn: http://www.linkedin.com/in/beiji-li\nGitHub: https://github.com/SuperiormonLBJ\nPortfolio: https://portfolio-beiji.vercel.app/  \n---'),
 Document(id='16a1d0e8-8c29-4917-932c-95c65a6c8b7d', metadata={'doc_type': 'personal', 'title': 'Hobbies & Personal Development', 'source': 'hobby.md', 'tags': '[hobby, basketball, diving, weightlifting, leadership, discipline, stress-management, growth-mindset]', 'importance': 'low', 'type': 'personal', 'year': '2026', 'section': '1. Basketball'}, page_content='project title: Hobbies & Personal Development\n\nsection: 1. Basketball\n\nCo

In [43]:
# Check all documents in the collection
collection = vectorstore._collection
all_docs = collection.get(include=["documents", "metadatas"])

# Count by source
from collections import Counter
sources = [meta.get("source", "unknown") for meta in all_docs["metadatas"]]
print("Documents by source:", Counter(sources))

# Check for duplicates
doc_contents = all_docs["documents"]
print(f"Total documents: {len(doc_contents)}")
print(f"Unique documents: {len(set(doc_contents))}")

Documents by source: Counter({'LLM-Based-Code-Reviewer.md': 18, 'NUS-Master-Project.md': 14, 'cv.md': 14, 'Prompt-Library.md': 13, 'Developer-Portal.md': 12, 'n8n.md': 12, 'Best-Price-Agent.md': 11, 'UOB-Software-Engineer.md': 10, 'Portfolio.md': 9, 'hobby.md': 6, 'NUS-Research-Engineer.md': 5})
Total documents: 124
Unique documents: 124


### RAG - Retrieval

In [44]:
from langchain_openai import ChatOpenAI


vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings, collection_name="my_collection")
retriever_similarity = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": TOP_K,
        "lambda_mult": 0.6  # tweak for diversity
    }
)

llm=ChatOpenAI(model=MODEL) # Langchain wrapper for OpenAI

System_prompt = """
You are a helpful assistant that can answer questions about the user's CV and hobbies.
You are given a question and a context.
You need to answer the question based on the context.
Context:
{context}
"""

In [45]:
from langchain_core.messages import HumanMessage, SystemMessage, convert_to_messages

import os
import sys
from pathlib import Path

# Add project root to path so we can import from utils
current_dir = Path(os.getcwd())
if (current_dir / "utils").exists():
    project_root = current_dir
elif (current_dir.parent / "utils").exists():
    project_root = current_dir.parent
else:
    project_root = current_dir.parent  # fallback

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.base_models import RerankOrder


def fetch_context(query):
    """
    Fetch the context for the query from the vector store with Top K results
    """
    context_similarity = retriever_similarity.invoke(query)
    context_mmr = retriever_mmr.invoke(query)
    context = context_similarity + context_mmr
    return context

SYSTEM_PROMPT_RERANK = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""

def rerank_documents(query: str, docs: list, top_k: int = TOP_K):
    """
    Rerank retrieved documents by relevance to the query using the LLM.
    Asks the LLM to return indices of the top_k most relevant passages (1-based).
    """
    user_prompt = f"The user has asked the following question:\n\n{query}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(docs):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_RERANK},
        {"role": "user", "content": user_prompt},
    ]
    llm_with_structured_output = llm.with_structured_output(RerankOrder)
    rerank_order = llm_with_structured_output.invoke(messages)
    reranked_docs = [docs[i-1] for i in rerank_order.order]


    print(f"Reranked order: {rerank_order.order}")
    return reranked_docs[:top_k]


def combine_all_user_questions(question: str, history: list) -> str:
    """
    Combine all user questions into a single question 
    to solve the issue that user ask "what did she do before"
    """
    prior = "\n".join(msg["content"] for msg in history if msg.get("role") == "user")
    
    # Handle empty prior case
    if prior:
        return prior + "\n" + question
    else:
        return question

def generate_answer(query, history: list):
    """
    Generate the answer for the query

    history from gradio is a list of dictionaries with keys "role" and "content" this is OpenAI format, need to convert it to the format that langchain can use
    """
    
    print(f"DEBUG - query type: {type(query)}, value: {query}")
    print(f"DEBUG - history type: {type(history)}, length: {len(history) if isinstance(history, list) else 'not a list'}")
    # ... rest of function

    # query = query["text"]
    
    # combine all user questions into a single question
    combined_question = combine_all_user_questions(query, history)
    context_docs = fetch_context(combined_question)
    # Rerank by relevance so the best passages are used first
    context_docs_reranked = rerank_documents(combined_question, context_docs, top_k=TOP_K)
    context = "\n".join([doc.page_content for doc in context_docs_reranked])

    # convert history to langchain format
    messages = [SystemMessage(content=System_prompt.format(context=context))]
    print("history:", history)
    print("convert_to_messages(history):", convert_to_messages(history))
    print("messages before extend:", messages)
    messages.extend(convert_to_messages(history)) # add history to the messages, convert_to_messages: convert the history to messages, from LangChain's format to OpenAI's format, import from langchain_core.messages
    print("messages after extend:", messages)
    messages.append(HumanMessage(content=query))
    print("messages after append human message:", messages)

    response = llm.invoke(messages)

    return response.content, context_docs_reranked

In [46]:
generate_answer("What is Beiji's current job based on CV?", [])

DEBUG - query type: <class 'str'>, value: What is Beiji's current job based on CV?
DEBUG - history type: <class 'list'>, length: 0
Reranked order: [5, 9, 3, 8, 4, 1, 10, 2, 7, 6]
history: []
convert_to_messages(history): []
messages before extend: [SystemMessage(content='\nYou are a helpful assistant that can answer questions about the user\'s CV and hobbies.\nYou are given a question and a context.\nYou need to answer the question based on the context.\nContext:\nproject title: UOB Software Engineer\n\nsection: 2. Software Engineer (Aug 2024 – Present)\n\n**Architecture**\n- Angular + Spring Boot (internal portal)\n- React + Node.js (Backstage plugins)  \n**Actions**\n- Built self-service developer workflows\n- Integrated:\n- CI/CD pipelines\n- Jira workflows\n- Service scaffolding  \n**Impact**\n- Served 200+ developers\n- Improved onboarding and deployment automation  \n**Signals**\nInternal Developer Platform (IDP)  \n---\nproject title: UOB Software Engineer\n\nsection: 2. Softwar

('Beiji is currently working as a Software Engineer at UOB since August 2024.',
 [Document(id='c8796938-1714-48d8-94f4-5714e7cbda0c', metadata={'doc_type': 'career', 'year': '2026', 'tags': '[software-engineer, devops, platform-engineering, ai-automation, internal-developer-platform, uob]', 'source': 'UOB-Software-Engineer.md', 'section': '2. Software Engineer (Aug 2024 – Present)', 'importance': 'high', 'subsection': '2.3 Internal Developer Portal / Backstage Integration', 'title': 'UOB Software Engineer', 'type': 'career'}, page_content='project title: UOB Software Engineer\n\nsection: 2. Software Engineer (Aug 2024 – Present)\n\n**Architecture**\n- Angular + Spring Boot (internal portal)\n- React + Node.js (Backstage plugins)  \n**Actions**\n- Built self-service developer workflows\n- Integrated:\n- CI/CD pipelines\n- Jira workflows\n- Service scaffolding  \n**Impact**\n- Served 200+ developers\n- Improved onboarding and deployment automation  \n**Signals**\nInternal Developer Platf

### Frontend with Gradio first for testing

In [ ]:
import gradio as gr
print(gr.__version__)

def format_context(context):
    result = "<h2 style='color: #ff7800;'>Relevant Context</h2>\n\n"
    for doc in context:
        result += f"<span style='color: #ff7800;'>Source: {doc.metadata['source']}</span>\n\n"
        result += doc.page_content + "\n\n"
    return result


def chat(history):
    """History is in messages format: [{"role": "user", "content": "..."}, ...]"""
    if not history:
        return history, ""
    
    last_message = history[-1]["content"]
    prior = history[:-1]
    answer, context = generate_answer(last_message, prior)
    
    # Create new history instead of modifying in place
    new_history = history + [{"role": "assistant", "content": answer}]
    return new_history, format_context(context)


def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

with gr.Blocks(title="Best Friend of Beiji", theme=theme) as ui:
    gr.Markdown("# 🏢 Best Friend of Beiji\nAsk me anything about Beiji!")

    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                    label="💬 Conversation", height=600, type="messages", show_copy_button=True
                )
            message = gr.Textbox(
                label="Your Question",
                placeholder="Ask anything about Beiji...",
                show_label=False,
            )

        with gr.Column(scale=1):
            context_markdown = gr.Markdown(
                label="📚 Retrieved Context",
                value="*Retrieved context will appear here*",
                container=True,
                height=600,
            )

    message.submit(
        put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]
    ).then(chat, inputs=chatbot, outputs=[chatbot, context_markdown])

ui.launch(inbrowser=True)

5.49.1
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


python(27564) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


DEBUG - query type: <class 'str'>, value: what is beiji's job
DEBUG - history type: <class 'list'>, length: 0
Reranked order: [4, 5, 3, 2, 1, 8, 6, 7, 9, 10]
history: []
convert_to_messages(history): []
messages before extend: [SystemMessage(content='\nYou are a helpful assistant that can answer questions about the user\'s CV and hobbies.\nYou are given a question and a context.\nYou need to answer the question based on the context.\nContext:\nproject title: NUS Research Engineer\n\nsection: 1. Role Overview\n\n**Position:** Research Engineer\n**Duration:** Sep 2021 – May 2023\n**Location:** On-site, Singapore  \nFocus:\n- Research in Intelligent Construction\n- Workflow optimization and automation for construction machinery\n- Traffic monitoring via computer vision  \n---\nproject title: UOB Software Engineer\n\nsection: 2. Software Engineer (Aug 2024 – Present)\n\n**Architecture**\n- Angular + Spring Boot (internal portal)\n- React + Node.js (Backstage plugins)  \n**Actions**\n- Buil